In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Create output folders if not exist
os.makedirs('../outputs/figures', exist_ok=True)

# Load fresh data
df = pd.read_csv('../data/student-mat.csv', sep=';')

# Create binary target column
df['pass'] = (df['G3'] >= 10).astype(int)

print("Data loaded successfully!")
print("Shape:", df.shape)

Data loaded successfully!
Shape: (395, 34)


In [2]:
# Drop G1, G2 (interim grades — cause data leakage)
# Drop G3 (raw grade — replaced by our 'pass' column)
df = df.drop(columns=['G1', 'G2', 'G3'])

print("Shape after dropping G1, G2, G3:", df.shape)
print("\nRemaining columns:")
print(df.columns.tolist())

Shape after dropping G1, G2, G3: (395, 31)

Remaining columns:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'pass']


In [3]:
# Separate categorical and numeric columns
categorical_cols = df.select_dtypes(include='str').columns.tolist()
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# Remove target from numeric list
numeric_cols.remove('pass')

print("Categorical columns:", len(categorical_cols))
print(categorical_cols)

print("\nNumeric columns:", len(numeric_cols))
print(numeric_cols)

Categorical columns: 17
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Numeric columns: 13
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']


In [4]:
# Convert text columns to numbers using one-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Shape after encoding:", df_encoded.shape)
print("\nAll columns after encoding:")
print(df_encoded.columns.tolist())

Shape after encoding: (395, 40)

All columns after encoding:
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'pass', 'school_MS', 'sex_M', 'address_U', 'famsize_LE3', 'Pstatus_T', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other', 'schoolsup_yes', 'famsup_yes', 'paid_yes', 'activities_yes', 'nursery_yes', 'higher_yes', 'internet_yes', 'romantic_yes']


In [5]:
# X = input features
# y = what we want to predict (pass or fail)
X = df_encoded.drop(columns=['pass'])
y = df_encoded['pass']

print("Features shape (X):", X.shape)
print("Target shape (y):  ", y.shape)

print("\nTarget distribution:")
print(y.value_counts())
print(f"\nPass rate: {y.mean()*100:.1f}%")

Features shape (X): (395, 39)
Target shape (y):   (395,)

Target distribution:
pass
1    265
0    130
Name: count, dtype: int64

Pass rate: 67.1%


In [6]:
# Clean Cell 6 — simple train test split, no SMOTE
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

Training set: (316, 39)
Testing set : (79, 39)


In [7]:
# Clean Cell 7 — scale numeric columns only
from sklearn.preprocessing import StandardScaler
import joblib

numeric_to_scale = ['age', 'Medu', 'Fedu', 'studytime', 'failures',
                    'famrel', 'freetime', 'goout', 'Dalc', 'Walc',
                    'health', 'absences']

sc = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[numeric_to_scale] = sc.fit_transform(
                                   X_train[numeric_to_scale])
X_test_scaled[numeric_to_scale]  = sc.transform(
                                   X_test[numeric_to_scale])

print("Scaling done!")

Scaling done!


In [8]:
# Clean Cell 8 — save files
import os
os.makedirs('../outputs', exist_ok=True)

X_train_scaled.to_csv('../outputs/X_train.csv', index=False)
X_test_scaled.to_csv('../outputs/X_test.csv',   index=False)
y_train.to_csv('../outputs/y_train.csv',         index=False)
y_test.to_csv('../outputs/y_test.csv',           index=False)
joblib.dump(sc, '../outputs/scaler.pkl')

print("Files saved!")
print("Training samples:", X_train_scaled.shape[0])
print("Testing samples :", X_test_scaled.shape[0])

Files saved!
Training samples: 316
Testing samples : 79


In [9]:
print("========== PREPROCESSING COMPLETE ==========")
print(f"Original training samples : 316")
print(f"Balanced training samples : {X_train_scaled.shape[0]}")
print(f"Testing samples           : {X_test_scaled.shape[0]}")
print(f"Pass rate after SMOTE     : {y_train.mean()*100:.1f}%")
print("============================================")

========== PREPROCESSING COMPLETE ==========
Original training samples : 316
Balanced training samples : 316
Testing samples           : 79
Pass rate after SMOTE     : 67.1%
